# 🏓 Entrenamiento de YOLO OBB (Oriented Bounding Boxes)

Este notebook está preparado para ejecutarse en **Lightning.ai** o cualquier entorno Jupyter. Permite descomprimir tu dataset en formato `.zip` y entrenar un modelo YOLO para detectar objetos orientados (rotados).

### 1. Instalación de Dependencias
Instalamos la librería de Ultralytics (YOLO) y otras dependencias necesarias.

In [ ]:
# Instalar Ultralytics
%pip install ultralytics roboflow matplotlib opencv-python numpy pyyaml

### 2. Descomprimir el Dataset desde el archivo `.zip`
Asegúrate de haber subido tu archivo `.zip` (descargado de Roboflow) al mismo directorio que este notebook en Lightning.ai.

**Ajusta el nombre del archivo `.zip` en la celda de abajo si es diferente.**

In [ ]:
import zipfile
import os
import glob

# Buscar cualquier archivo .zip en el directorio actual
zip_files = glob.glob("*.zip")

if zip_files:
    # Usar el primer zip que encuentre, o puedes escribir el nombre exacto aquí:
    ZIP_NAME = zip_files[0] 
    EXTRACT_DIR = "./dataset_obb"

    print(f"Descomprimiendo {ZIP_NAME} en {EXTRACT_DIR}...")
    with zipfile.ZipFile(ZIP_NAME, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("¡Descompresión completada con éxito!")
    
    # Mostrar el contenido extraído
    print("Contenido:", os.listdir(EXTRACT_DIR))
else:
    print("❌ ERROR: No se encontró ningún archivo .zip en el directorio actual. Por favor, sube tu dataset.")

### 3. Verificar y corregir rutas en `data.yaml`
Roboflow suele exportar rutas absolutas que pueden no coincidir con el entorno de Lightning.ai. Ejecutamos este bloque para asegurarnos de que las rutas relativas sean correctas.

In [ ]:
import yaml
import os

yaml_path = "./dataset_obb/data.yaml"
EXTRACT_DIR = "./dataset_obb"

if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    
    # Ajustamos las rutas de entrenamiento y validación para que sean relativas a la carpeta extraída
    data['path'] = os.path.abspath(EXTRACT_DIR)
    data['train'] = 'train/images'
    data['val'] = 'valid/images'
    if 'test' in data:
        data['test'] = 'test/images'
        
    with open(yaml_path, 'w') as f:
        yaml.safe_dump(data, f, default_flow_style=False)
        
    print("✅ Archivo data.yaml actualizado con éxito:")
    with open(yaml_path, 'r') as f:
        print(f.read())
else:
    print("❌ ERROR: No se encontró el archivo data.yaml en la carpeta especificada.")

### 4. Configurar y Entrenar el Modelo YOLO OBB
Usamos `yolov8n-obb.pt` (o `yolo11n-obb.pt`) como modelo base. Estos modelos están entrenados para predecir cajas orientadas (con rotación).

In [ ]:
from ultralytics import YOLO

# 1. Cargar el modelo OBB preentrenado
model = YOLO('yolov8n-obb.pt') 

# 2. Entrenar el modelo
results = model.train(
    data      = './dataset_obb/data.yaml', # Ruta al data.yaml que corregimos
    epochs    = 300,                        # 300 épocas solicitadas
    patience  = 50,                         # Early stopping con 50 épocas de paciencia
    imgsz     = 640,                        # Tamaño de imagen
    batch     = 16,                         # Tamaño de lote
    optimizer = 'AdamW',                    # Recomendado para datasets pequeños
    lr0       = 0.001,
    
    # Aumentación de datos
    mosaic    = 1.0,
    mixup     = 0.1,
    degrees   = 15.0,
    translate = 0.1,
    scale     = 0.5,
    fliplr    = 0.5,
    
    # Configuración de guardado
    project   = './runs',
    name      = 'tenis_mesa_obb_run',
    exist_ok  = True,
    save      = True,                       # Asegura que se guarde el mejor modelo (best.pt)
    plots     = True
)

### 5. Validar el Modelo
Una vez completado el entrenamiento, podemos validar las métricas obtenidas con el mejor modelo guardado.

In [ ]:
# Cargar el mejor modelo entrenado
best_model = YOLO('./runs/tenis_mesa_obb_run/weights/best.pt')

# Validar
metrics = best_model.val()
print("Métricas de validación calculadas con éxito.")